# Power Analysis — TXB_23
## Base de Dados Social | Events Page CTA Test

We calculate the minimum sample size required to detect a meaningful effect
for our two primary KPIs, using the pre/control group as baseline.

In [1]:
import pandas as pd
import numpy as np
from scipy import stats
import math
from io import StringIO

with open('../data/TXB_23_landingpage(in).csv', 'rb') as f:
    raw = f.read()

lines = raw.split(b'\r\n')
cleaned = []
for line in lines:
    line = line.strip()
    if line.startswith(b'"') and line.endswith(b'"'):
        line = line[1:-1]
    line = line.replace(b'""', b'"')
    cleaned.append(line.decode('utf-8', errors='replace'))

df = pd.read_csv(StringIO('\n'.join(cleaned)))
print(df.shape)

(1163, 16)


## Baseline Metrics

We use the pre/control group to establish baseline conversion rates for our two KPIs:
- **cta_click** (kpi_y) — binary, whether the visitor clicked the CTA button
- **event_enrollment** (kpi_x) — continuous, engagement score

In [2]:
control = df[df['arm'] == 'pre'].copy()
control = control.rename(columns={'kpi_x': 'event_enrollment', 'kpi_y': 'cta_click'})

print(f"Control group size: {len(control)}")
print(f"CTA Click baseline rate: {control['cta_click'].mean():.4f}")
print(f"Event Enrollment baseline mean: {control['event_enrollment'].mean():.2f}")
print(f"Event Enrollment baseline std: {control['event_enrollment'].std():.2f}")

Control group size: 349
CTA Click baseline rate: 0.0888
Event Enrollment baseline mean: 30.60
Event Enrollment baseline std: 28.12


## Sample Size Calculation

Using the formula for two-sided tests with power=0.80 and α=0.05.
Target rates are set based on the minimum detectable effect we care about.

In [3]:
# CTA Click (binary)
cta_mu_0 = control['cta_click'].mean()
cta_mu_1 = 0.12
cta_sigma = math.sqrt(cta_mu_0 * (1 - cta_mu_0))
cta_effect_size = cta_mu_1 - cta_mu_0

# Event Enrollment (continuous)
event_mu_0 = control['event_enrollment'].mean()
event_mu_1 = 34.6
event_sigma = control['event_enrollment'].std()
event_effect_size = event_mu_1 - event_mu_0

# Power and significance
power = 0.80
alpha = 0.05
z_alpha_2 = stats.norm.ppf(1 - alpha / 2)
z_beta = stats.norm.ppf(power)

# Sample sizes
cta_n = math.ceil(((z_alpha_2 + z_beta)**2 * 2 * cta_sigma**2) / cta_effect_size**2)
event_n = math.ceil(((z_alpha_2 + z_beta)**2 * 2 * event_sigma**2) / event_effect_size**2)

print(f"CTA Click — baseline: {cta_mu_0:.4f}, target: {cta_mu_1}, required n per group: {cta_n}")
print(f"Event Enrollment — baseline: {event_mu_0:.2f}, target: {event_mu_1}, required n per group: {event_n}")

CTA Click — baseline: 0.0888, target: 0.12, required n per group: 1308
Event Enrollment — baseline: 30.60, target: 34.6, required n per group: 776


## Power Analysis Summary

| KPI | Baseline | Target | Required n/group | Actual n (pre) | Actual n (treatment) | Powered? |
|-----|----------|--------|-----------------|----------------|----------------------|----------|
| CTA Click (kpi_y) | 0.0888 | 0.12 | 1,308 | 349 | 814 |  No |
| Event Enrollment (kpi_x) | 30.60 | 34.6 | 776 | 349 | 814 |  No |

Both KPIs are **underpowered** given our sample sizes. This means our test may not have 
enough statistical power to reliably detect the desired effect sizes. Results should 
be interpreted with caution.